<a href="https://colab.research.google.com/github/ColeUmland/DSC-420-Midterm-Kaggle-Competition/blob/main/submission_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e10_path = kagglehub.competition_download('playground-series-s6e10')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Introduction
Prediction Problem: We are trying to if a passenger will be satisfied with their flight using several predictor columns. The dataset contains almost 700,000 observations. There are 18 available variables.

Response Variable: Our response variable is `satisfaction`. We want to know if we can predict the satisfaction a passenger will have based on certain in-flight characteristics.

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/test.csv")

# EDA
train.describe()

# Data Exploration
The dataset contains both numerical and categorical variables. The numerical variables include passenger information such as age and flight distance, as well as service ratings and departure and arrival delays. The categorical variables include gender, customer type, type of travel, class, and satisfaction. The summary statistics for the numerical variables show their ranges, means, and standard deviations, which helps identify the typical passenger and possible unusual values in the data.

In [ ]:
# Table of Missing Value counts
missing_table = pd.DataFrame({
    "Missing Values": train.isnull().sum(),
    "Percent Missing": train.isnull().mean() * 100
})
missing_table

# Missing Values
The only predictor with missing values was `Arrival Delay in Minutes`. To fill these 292 missing values, we calculated the average difference between arrival delay and departure delay for observations where both values were available. I then added this average difference to each passenger's departure delay to estimate their missing arrival delay. I chose this method because arrival and departure delays are strongly related, so using the passenger's actual departure delay provides a more reasonable estimate than just replacing every missing value with the overall median.

In [ ]:
# Handling Missing Values
# Calculate the mean delay difference using training data only
delay_difference = (
    train["Arrival Delay in Minutes"] -
    train["Departure Delay in Minutes"]
).mean()

# Apply the same missing-value handling to train and test
for df in [train, test]:
    df["Arrival Delay in Minutes"] = df[
        "Arrival Delay in Minutes"
    ].fillna(
        df["Departure Delay in Minutes"] + delay_difference
    )

# Distribution and Correlation Matrix Findings
Distribution: The response variable is categorical, so either satisfied or not satisfied. This calls for classification models

Correlation Matrix Findings: Several predictors appear to be correlated with one another. `Departure Delay in Minutes` and `Arrival Delay in Minutes`. This is an example of two predictors telling a similar story. We might run into multicollinearity here.

- When it comes to correlation with our response variable, `satisfaction`, it seems `Flight Distance`, `Online boarding`, and `Inflight entertainment` have positive correlations with our response variable. We'll come back to this when we are choosing our predictors.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

####### 2. Response Distribution & Correlation Plot of All Predictors ############

# --- A. Response Distribution ---
plt.figure(figsize=(8, 5))
# Pass x as a string column name when specifying data=train
sns.histplot(data=train, x='satisfaction', kde=True, bins=30, color='skyblue', edgecolor='black')

plt.title('Satisfaction Distribution')
plt.xlabel('Satisfaction Score')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


# --- B. Correlation Plot of All Predictors ---
plt.figure(figsize=(10, 8))
# Calculate Pearson correlation matrix for numeric columns
corr_matrix = train.corr(numeric_only=True)

# Heatmap visualization
sns.heatmap(
    corr_matrix,
    annot=True,         # Display numerical values in cells
    fmt='.2f',          # Format numbers to 2 decimal places
    cmap='coolwarm',    # Color palette (blue = negative, red = positive correlation)
    vmin=-1, vmax=1,    # Scale range
    linewidths=0.5
)

plt.title('Correlation Matrix of All Predictors')
plt.tight_layout()
plt.show()

# Exploratory Data Analysis

Dropping Unessecary Predictors: `id` is essentialy just a row count in this example and will have no predictive power besides noise. Therefore we drop the `id` column from our model.

# Handling Skewed Delays
Since our delay columns (`Arrival Delay` & `Departure Delay`) are very right skewed. That is, they have a median close to zero with a mean that is pulled up by the very large delay outliers. In an effort to standardize these, we are going to take the loge of them. Now they will appear more linear.

# Feature Engineering
`Departure Delay` and `Arrival Delay` seem to be correlated via the correlation matrix above. If your plane takes off late, there is a great chance it will land late as well. By adding an interaction term, our model can account for this covariance.

# Composite Survey Scores
In the correlation matrix, the in-flight survey scores seem to correlate moderately. All of these variables are on a Likert scale. So, to account for this correlation, we created an interaction term between all of these in-flight survey variables to account for their covariance. This was done by taking the average of all of the survey scores and putting them into a new metric.

In [ ]:
################ 3 #################
def preprocess_airline_data(df):
    df = df.copy()

    # 1. Dropping the `id` column. It isn't applicable in our model
    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    # 2. Handle Skewed Delays & Feature Engineering
    # Since our delay columns are very heavily right-skewed, we are taking the log of those two columns as an effort to standardize them
    # Then we are creating an interaction term between departure and arrival delay as they are deemed to be very correlated in our correlation heat map above.
    # This is done after the log is taken
    df['Departure Delay Log'] = np.log1p(df['Departure Delay in Minutes'])
    df['Arrival Delay Log'] = np.log1p(df['Arrival Delay in Minutes'])
    df['Total_Delay'] = df['Departure Delay Log'] + df['Arrival Delay Log']

    # 3 Composite Survey Scores
    # Taking several correlated "inflight" rating metrics and combining them
    # In the heat map, they were seen to correlate amongst themselves
    # Creating a new column that takes the average of each row's score across the metrics into one metric
    inflight_cols = ['Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Cleanliness', 'Food and drink']
    df['Inflight_Score_Avg'] = df[inflight_cols].mean(axis=1)

    # Returning the altered data frame at the end of this function
    return df

# Usage:
train_cleaned = preprocess_airline_data(train)
# Updating the test set as well
test = preprocess_airline_data(test)
train_cleaned.describe()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = train_cleaned.drop(columns=["satisfaction"])
y = train_cleaned["satisfaction"]

# Encode the response variable
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Identify numerical and categorical predictors
num_cols = X.select_dtypes(include=np.number).columns.tolist()

cat_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Split the data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Numerical predictors:", len(num_cols))
print("Categorical predictors:", len(cat_cols))

# Logistic Regression Train & Test Split

In [ ]:
# Step 4: Logistic Regression

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

logistic_probs = logistic_model.predict_proba(X_valid)[:, 1]
logistic_auc = roc_auc_score(y_valid, logistic_probs)

print("Logistic Regression ROC AUC:", round(logistic_auc, 5))

# Logistic Regression CV

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV

# Define 5-fold cross-validation (Stratified for classification)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define hyperparameters to search for Logistic Regression
logistic_param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__solver": ["lbfgs", "liblinear"]
}

# Set up GridSearchCV using the same cross-validation object (cv)
logistic_grid_search = GridSearchCV(
    estimator=logistic_model,
    param_grid=logistic_param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

logistic_grid_search.fit(X_train, y_train)

print("Best Logistic Parameters:", logistic_grid_search.best_params_)
print("Best Logistic CV ROC AUC:", round(logistic_grid_search.best_score_, 5))

# Random Forest Train / Test Split

In [ ]:
# Random Forest Model

from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=50,
        max_depth=12,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

rf_probs = random_forest_model.predict_proba(X_valid)[:, 1]
rf_auc = roc_auc_score(y_valid, rf_probs)

print("Random Forest ROC AUC:", round(rf_auc, 5))

# Random Forest CV

In [ ]:
# Define hyperparameters to search
rf_param_grid = {
    "model__n_estimators": [50],
    "model__max_depth": [8, 12],
    "model__min_samples_leaf": [2, 5, 10],
    "model__max_features": ["sqrt", "log2"]
}

# Set up GridSearchCV
rf_grid_search = GridSearchCV(
    estimator=random_forest_model,
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# Fit on training data (cross-validation happens internally here)
rf_grid_search.fit(X_train, y_train)

print("Best RF Parameters:", rf_grid_search.best_params_)
print("Best RF CV ROC AUC:", round(rf_grid_search.best_score_, 5))

# CATBoost Train & Test Split

In [ ]:
# CatBoost Gradient Booster

from catboost import CatBoostClassifier

catboost_model = CatBoostClassifier(
    iterations=1000,
    depth=7,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    allow_writing_files=False,
    thread_count=-1
)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid)
)

catboost_probs = catboost_model.predict_proba(X_valid)[:, 1]
catboost_auc = roc_auc_score(y_valid, catboost_probs)

print("CatBoost ROC AUC:", round(catboost_auc, 5))

# CATBoost CV

In [ ]:
# Complete the CatBoost pipeline
catboost_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", CatBoostClassifier(
        random_seed=42,
        verbose=0,
        thread_count=-1
    ))
])

# Define hyperparameters to search for CatBoost
cat_param_grid = {
    "model__iterations": [50],
    "model__depth": [6, 8, 12],
    "model__learning_rate": [0.01, 0.05, 0.1, .2]
}

# Set up GridSearchCV
cat_grid_search = GridSearchCV(
    estimator=catboost_model,
    param_grid=cat_param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

cat_grid_search.fit(X_train, y_train)

print("Best CatBoost Parameters:", cat_grid_search.best_params_)
print("Best CatBoost CV ROC AUC:", round(cat_grid_search.best_score_, 5))

# Model Comparison
|Model|Test Split AUC | K-Fold CV TEST AUC |
|---|---|---|
|Logistic Regression | .9206 | .9244 |
|Random Forests | .95413 | .95542 |
|CatBoost | .9575 | .9575 |

Methods
- We initially completed a train a test split to compare AUC's across the models. The AUC's are shown above.
- Next, to better tune the parameters of our models, we completed K-Fold CV on the training set where K = 5. We hypothesized this would beter tune our models as resampling via CV would give the models more opportunites to train.

- All of our findings below will be discussed in terms of our K-Fold CV AUC's

Which model perfomred the best on the validation set?
- The CatBoost model had the best performance with an AUC of .9575. Interpretting this, if our model was given a pair of observations, one satisfied passenger and one dissatisfied passenger, the model would give a higher likelihood of being satisfied the the truely satisfied passenger 95.75% of the time.

The Flexible Models outperformed the less flexible model.
- The CatBoost and Random forests designs are an ensemble approach.Since these both did better than logistic regression, it can be inferred that our data is non-linear and has many interactions.

CatBoost did slightly better than Random Forrests
- CatBoost beat our Random Forests by .00233 in AUC. Random Forrests' bagging was beat out by CatBoost's gradient boosting.

# Identify Important Predictors

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from catboost import Pool

# Extract the best fitted pipelines from your GridSearchCV objects
best_logistic_pipe = logistic_grid_search.best_estimator_
best_rf_pipe = rf_grid_search.best_estimator_
best_cat_pipe = cat_grid_search.best_estimator_

############ Significant Predictors: Logistic Regression ################
X_train_processed = best_logistic_pipe.named_steps['preprocessor'] if 'preprocessor' in best_logistic_pipe.named_steps else best_logistic_pipe.named_steps['preprocessor']
# Better yet, directly access via the named preprocessor step:
X_train_processed = best_logistic_pipe.named_steps['preprocessor'].transform(X_train)

# Get feature names from the preprocessor pipeline
if hasattr(best_logistic_pipe.named_steps['preprocessor'], 'get_feature_names_out'):
    feature_names = best_logistic_pipe.named_steps['preprocessor'].get_feature_names_out()
else:
    feature_names = [f"feat_{i}" for i in range(X_train_processed.shape[1])]

# Convert y_train to a pandas Series
y_train_series = pd.Series(y_train).reset_index(drop=True)

# Fit statsmodels Logit for p-values
X_train_const = sm.add_constant(pd.DataFrame(X_train_processed, columns=feature_names))
logit_fit = sm.Logit(y_train_series, X_train_const).fit(disp=False)

logit_df = pd.DataFrame({
    'Feature': X_train_const.columns,
    'Logit_PValue': logit_fit.pvalues
}).query("Feature != 'const'")

logit_df['Logit_Sig'] = logit_df['Logit_PValue'] < 0.05

########### Extract Feature Importance: Random Forest ##############
rf_fitted_model = best_rf_pipe.named_steps['model']

rf_df = pd.DataFrame({
    'Feature': feature_names,
    'RF_Importance': rf_fitted_model.feature_importances_
})
rf_df['RF_Sig'] = rf_df['RF_Importance'] > rf_df['RF_Importance'].mean()

############### Extract Feature Importance: CatBoost ##################
cb_fitted_model = best_cat_pipe.named_steps['model']

# Wrap the processed array into a CatBoost Pool with correct feature names
train_pool = Pool(data=X_train_processed, feature_names=list(feature_names))

cb_df = pd.DataFrame({
    'Feature': feature_names,
    'CatBoost_Importance': cb_fitted_model.get_feature_importance(data=train_pool)
})
cb_df['CatBoost_Sig'] = cb_df['CatBoost_Importance'] > cb_df['CatBoost_Importance'].mean()

# Print out results
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("--- Logistic Regression Significant Features (p < 0.05) ---")
print(logit_df[logit_df['Logit_Sig']].sort_values(by='Logit_PValue').head(10))

print("\n--- Top Random Forest Important Features ---")
print(rf_df.sort_values(by='RF_Importance', ascending=False).head(10))

print("\n--- Top CatBoost Important Features ---")
print(cb_df.sort_values(by='CatBoost_Importance', ascending=False).head(10))

# Significant Predictor Findings

- The following predictors were deemed as significant in all three of our models
|Predictor|Logistic Regression P-Value | Random Forests Importance | CatBoost Importance |
|---|---|---|---|
|Inflight Wifi Service| 0|0.093|19.251|
|Online Boarding | 0 | .206 | 9.056|

 - Only two of the predictors used were found as 'significant' in all three of the models. Inflight Wifi Service and Online Boarding are those two.
- This suggests high variability in variable significance. With this information we can infer that some of the predictors may be noise. Removing these from the logistic regression model might improve our performance there. BUT, removing these from the ensemble models might hinder their succes because there would be less predictors to improve our model upon through node splits and gradient boosting.

# What our Models Suggests

Internet Ease and Feasibility
- Our models suggest that wifi and and the online boarding process drive passenger satisfaction. Passengers prioritize digital seemlessness. Because the predictors are varying across models, and these are the only two predictors that are included in all models, we can confidently say these significant predictors of passenger satisifaction.

Airline Takeaway
- Airlines should invest in their digital experiences to improve their passenger satisfaction scores. Things like app efficiency and readability, or internet speeds on the plane are confident avenues airlines can invest in to improve passenger experience.

# Submitting to Competition Steps

In [ ]:
# 1. Load the sample submission file
sample_submission = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv"
)

# 2. Create test features using the same preprocessing
X_test = test.drop(columns=["satisfaction"], errors="ignore")

# 3. Predict probabilities using the best fitted pipeline (handles preprocessing automatically)
test_probs = best_cat_pipe.predict_proba(X_test)[:, 1]

# 4. Fill the required submission columns
submission = sample_submission.copy()
submission["satisfaction"] = test_probs

# 5. Verify the submission
print("Submission columns:", submission.columns.tolist())
print("Submission shape:", submission.shape)
print(submission.head())
print("Missing predictions:", submission["satisfaction"].isna().sum())

# 6. Save the CSV file
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("CSV saved successfully!")

# Reflection
- An interesting finding is how the best parameters for the CATBoost CV model performed worse on the test set than our train test split CATBoost model did. You would expect that with more data to train on, a boosting method would perform better.
- One thing that worked well in this project was the data cleaning and feature engineering. Handling the missing arrival delay values and creating features such as the average difference between departure delay and arrival delay gave the models more useful information to work with.
- Logistic regression also worked well as a baseline because it gave us a simple model to compare the other models against.
- Random forest and CatBoost allowed us to test models that could capture more complicated relationships in the data. One thing that did not work as well was the runtime of the more complex models. We shortened the CatBoost training by reducing the number of iterations and tree depth.
- Overall, comparing the three models helped us see how different machine learning approaches performed on the same problem.
- If we continued the project, we would try tuning the models better, testing different hyperparameters, and possibly creating additional features from the passenger ratings and delay variables.
- We would also try other top models to see if we could improve the ROC AUC and compete for a top spot on the leaderboard.